In [ ]:
# cmd
# python -m venv .venv
# Set-ExecutionPolicy RemoteSigned
# .\.venv\Scripts\activate

In [ ]:
%pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
print("드라이버 실행 완료")

드라이버 실행 완료


In [ ]:
main_url = "https://www.oliveyoung.co.kr/store/main/getBestList.do?dispCatNo=900000100100001&fltDispCatNo=&pageIdx=1&rowsPerPage=8" # 스크래핑할 페이지 URL 저장
driver.get(main_url)

wait = WebDriverWait(driver, 300)

wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.best-area")))
time.sleep(2)

soup = BeautifulSoup(driver.page_source, "html.parser")
items = soup.select("#Container > div.best-area > div.TabsConts.on > ul > li")
sub_url = [item.select_one("div.prd_info > a.prd_thumb")["href"] for item in items[:10]]
print(f"수집된 링크 {len(sub_url)}개")

for i, url in enumerate(sub_url, 1):
    print(f"{i}. {url.split('goodsNo=')[1].split('&')[0]}")

수집된 링크 10개
1. A000000223414
2. A000000206698
3. A000000226498
4. A000000247641
5. A000000217607
6. A000000171427
7. A000000247193
8. A000000247728
9. A000000218845
10. A000000171423


In [ ]:
result = []

for i, url in enumerate(sub_url, 1):
    print(f"[{i}/10] 진행 중...")
    driver.get(url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    try:
        brand = soup.select_one('[class*="_btn-brand"]').get_text(strip=True)
    except:
        brand = "없음"

    try:
        name = soup.select_one('div[class*="GoodsDetailInfo_title"] h3').get_text(strip=True)
    except:
        name = "없음"

    try:
        breadcrumb = soup.select_one('div[class*="Breadcrumb_breadcrumb-inner"]')
        links = breadcrumb.select('a[role="link"]')
        category = ">".join([a.get_text(strip=True) for a in links])
    except:
        category = "없음"

    try:
        price = soup.select_one('[data-qa-name="text-product-original-price"] span:first-child').get_text(strip=True) + "원"
    except:
        price = "없음"

    try:
        discount = soup.select_one('[data-qa-name="text-product-discount-price"] span:first-child').get_text(strip=True) + "원"
    except:
        discount = "없음"

    try:
        review_count = soup.select_one('div[class*="review-count"] button span').get_text(strip=True) + "건"
    except:
        review_count = "없음"

    try:
        rating_tag = soup.select_one('span.rating')
        for blind in rating_tag('.oyblind'):
            blind.decompose()
        rating = rating_tag.get_text(strip=True)
    except:
        rating = "없음"

    result.append([i, brand, name, category, price, discount, rating, review_count])

[1/10] 크롤링 중...
[2/10] 크롤링 중...
[3/10] 크롤링 중...
[4/10] 크롤링 중...
[5/10] 크롤링 중...
[6/10] 크롤링 중...
[7/10] 크롤링 중...
[8/10] 크롤링 중...
[9/10] 크롤링 중...
[10/10] 크롤링 중...


In [ ]:
columns = ["순위", "브랜드", "상품명", "카테고리", "정가", "할인가", "평점", "리뷰건수"]
df = pd.DataFrame(result, columns=columns)
df

,순위,브랜드,상품명,카테고리,정가,할인가,평점,리뷰건수
0,1,메디힐,[3월 올영픽/15년연속 1위] 메디힐 에센셜 마스크팩 10+1/10매 기획 7종 ...,마스크팩>시트팩>시트 마스크,"20,000원","10,000원",평점4.9,"356,842건"
1,2,빌리프,[3월올영픽/생기충전][대용량] 빌리프X바나나킥 비타C 토닝 세럼 50ml 기획 (...,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"44,000원","33,000원",평점4.9,"2,782건"
2,3,메디큐브,[3월 올영픽/1등미백앰플] 메디큐브 PDRN 핑크 앰플 30ml 기획 (본품+리필...,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"46,000원","24,600원",평점4.7,"20,262건"
3,4,오아드,[3월올영픽/허그유어스킨콜라보 증정기획] 오아드 슬레인 파우더믹싱 워터틴트 8종 단...,메이크업>립메이크업>립틴트,"23,000원","14,800원",평점4.5,647건
4,5,밀잇,[1+1] 밀잇 식단관리 단백질쉐이크 40g 7종 택1,푸드>식단관리/이너뷰티>프로틴 쉐이크,없음,"3,900원",평점4.8,"45,564건"
5,6,메디힐,[3월 올영픽/1위패드] 메디힐 더마 패드 200매 대용량 기획 세트 7종 골라담기,마스크팩>패드>패드,"39,900원","28,900원",평점4.8,"70,101건"
6,7,토리든,[3월올영픽/1위세럼]토리든 다이브인 저분자 히알루론산 세럼 50ml 한정 리필 기획,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"36,000원","27,000원",평점4.8,"72,479건"
7,8,딜라이트 프로젝트,[3월 올영픽] 딜라이트 프로젝트 베이글칩 먼작귀 에디션 8종 택1 (빅띠부씰 랜덤...,푸드>식단관리/이너뷰티>헬시 스낵,없음,"2,700원",평점4.9,"2,219건"
8,9,릴리이브,[3월올영픽/윤은혜PICK/대용량]릴리이브 그로우턴 엑소좀 브러쉬 앰플 130ml ...,헤어케어>두피앰플/토닉>헤어토닉/두피토닉,"34,800원","29,900원",평점4.8,"8,680건"
9,10,어노브,[3월 올영픽 한정기획] 어노브 딥 데미지 헤어 트리트먼트 EX 320ml 벚꽃/더...,헤어케어>트리트먼트/팩>헤어 트리트먼트,"42,000원","29,800원",평점4.9,"80,892건"


In [ ]:
df.to_excel("olive_young-rank-top-10.xlsx", index=False)
print("엑셀 파일 저장 완료 - olive_young-rank-top-10.xlsx")

엑셀 파일 저장 완료 - olive_young-rank-top-10.xlsx
